In [ ]:
#!/usr/bin/env python
# coding: utf-8

"""
Refactored script for Laue Patterns digital processing.

Objectives:
- Load and extract pixel intensities or ROI statistics.
- Visualize 2D/1D scalar profiles (intensity, spot position, etc.).
- Support workflows for 2D maps (mesh scans) and 1D scans (DAXM, ascan).

Author: J.-S. Micha
Last Revision: July 2026
"""
%matplotlib widget

import os
import time
import copy
import glob
import logging
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
import h5py
import fabio
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import FloatProgress, IntSlider, Button, HBox, VBox, Output
#from tqdm import tqdm
from tqdm.notebook import tqdm
import multiprocessing
from multiprocessing import Pool, cpu_count, active_children

# LaueTools imports
import LaueTools as LT
import LaueTools.GUI.mosaic as MOS
import LaueTools.generaltools as GT
import LaueTools.IOimagefile as IOimage
import LaueTools.imageprocessing as Improc
import LaueTools.dict_LaueTools as DictLT
import LaueTools.IOLaueTools as IOLT
import LaueTools.readmccd as RMCCD
import LaueTools.blissdatafolderstructure as bf
import LaueTools.logfile_reader as iohdf5
import LaueTools.blissscan as bscan
import LaueTools.scripts.workflows as wf

In [ ]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filename="pixel_monitoring.log"
)
logger = logging.getLogger(__name__)

In [ ]:
# ============================================================================
# CONFIGURATION MANAGEMENT
# ============================================================================

class ConfigManager:
    """Manages configuration parameters for the experiment."""

    def __init__(
        self,
        exp_id: str,
        data_at_esrf: bool = True,
        hdf5_logfile_exists: bool = True,
        ccd_label: str = "EIGER_4MCdTe",
        on_linux: bool = True,
        jupyter_lab: bool = True,
    ):
        self.exp_id = exp_id
        self.data_at_esrf = data_at_esrf
        self.hdf5_logfile_exists = hdf5_logfile_exists
        self.ccd_label = ccd_label
        self.on_linux = on_linux
        self.jupyter_lab = jupyter_lab
        self.experiment_folder = self._set_experiment_folder()

    def _set_experiment_folder(self) -> str:
        """Set the experiment folder path based on configuration."""
        if self.data_at_esrf:
            nice_folder = "visitor"
            experiment_folder = os.path.join("/data", nice_folder, f"{self.exp_id}/bm32/")
            if os.path.exists(experiment_folder):
                experiment_folder = bf.setExperimentFolder_with_date(experiment_folder)
                logger.info(f"Experiment folder set to: {experiment_folder}")
            else:
                raise FileNotFoundError(f"Experiment folder not found: {experiment_folder}")
        else:
            experiment_folder = "/my/folder/to/data"
        return experiment_folder

# workflow classes

In [ ]:
from LaueTools.imagescollector import (
    collectpixelvalue_singlefile,
    collectroissum_singlefile,
    collectroisptp_singlefile,
    collectroiarray_singlefile)

# ============================================================================
# WORKFLOW CLASSES
# ============================================================================

## Local definition: Mosaic workflow

In [ ]:

def _collect_roi_array_single_file_wrapper(args):
    """Wrapper function to unpack args for _collect_roi_array_single_file."""
    return MosaicWorkflow._collect_roi_array_single_file(*args)

class MosaicWorkflow:
    """Handles mosaic-related workflows for 2D maps."""

    def __init__(self, params: Dict[str, Any]):
        self.params = params
        self.roicenter = params.get("roicenter")
        self.boxsize_X = params.get("boxsize_X", 19)
        self.boxsize_Y = params.get("boxsize_Y", 19)
        self.collector = params.get("collector", "mosaic")

    def validate_roicenter(self) -> bool:
        """Validate that the ROI center is within detector bounds."""
        if self.roicenter is None:
            logger.error("ROI center is not set.")
            return False
        x_roi, y_roi = self.roicenter
        max_dimension = 2015  # Adjust based on detector specs
        if (
            x_roi - self.boxsize_X < 0
            or x_roi + self.boxsize_X > max_dimension
            or y_roi - self.boxsize_Y < 0
            or y_roi + self.boxsize_Y > max_dimension
        ):
            logger.error(
                f"ROI center {self.roicenter} is too close to the detector border for boxsize: {(self.boxsize_X, self.boxsize_Y)}"
            )
            return False
        return True

    def run_mosaic_workflow(
        self, nb_cpus: int = 64, list_indices: Optional[List[int]] = None
    ) -> np.ndarray:
        """
        Run the mosaic workflow to collect pixel intensities.
        """
        if not self.validate_roicenter():
            raise ValueError("Invalid ROI center or boxsize.")
    
        folder = self.params.get("folder")
        prefix = self.params.get("prefix")
        ccd_label = self.params.get("CCDLabel")
    
        if list_indices is None:
            list_indices = self.params.get("listindices")
    
        nb_images = len(list_indices)
        max_nb_cpus = cpu_count()
        nb_cpus = min(nb_cpus, max_nb_cpus)
        logger.info(f"Using {nb_cpus} CPUs for {nb_images} images.")
    
        # Prepare arguments for multiprocessing
        args_mosaic = zip(
            list_indices,
            itertools.repeat(self.roicenter),
            itertools.repeat(prefix),
            itertools.repeat(folder),
            itertools.repeat(self.boxsize_X),
            itertools.repeat(self.boxsize_Y),
            itertools.repeat(ccd_label),
        )
    
        # Use the module-level wrapper function
        with Pool(nb_cpus) as pool:
            all_results = list(
                tqdm(
                    pool.imap(
                        _collect_roi_array_single_file_wrapper,  # Use the module-level wrapper
                        args_mosaic,
                    ),
                    total=nb_images,
                    desc="Mosaic collection progress",
                )
            )
    
        self.all_results = np.array(all_results)
        return self.all_results
    
    def _arrange2D(self):
        """rerrange all_results in 2D  
        
        return: mosaic (2D array of small extracted images), bigimage (single image of the mosaic of all extracted small images)"""
        scantype = self.params.get("scantype")
        mapdimensions = self.params.get("mapdimensions")
        listindices = self.params.get("listindices")
        all_results= self.all_results

        boxsize_X = self.boxsize_X
        boxsize_Y = self.boxsize_Y
        
        if scantype=='map':
            # processing and rearranging collected ROIs imagelets
            dimfast, dimslow = mapdimensions
            #print('axis dimensions: dimslow, dimfast',dimslow, dimfast)
            mosaic = np.zeros((dimslow, dimfast, 2*boxsize_Y+1, 2*boxsize_X+1))
            #print('mosaic.shape', mosaic.shape)
            dict_map_imageindex ={}
        
            #print(d['listindices'])
            
            sm = mosaic.shape
            bigimage = np.zeros((sm[0]*sm[2],sm[1]*sm[3]))
            
            
            if dimfast > 0:
                for map_imageindex, absolute_imageindex in enumerate(listindices):
                    
                    imap, jmap = map_imageindex // dimfast, map_imageindex % dimfast
            
                    dict_map_imageindex[map_imageindex] = [absolute_imageindex,
                                                            map_imageindex,
                                                            imap, jmap]
                    
                    raw = all_results[map_imageindex,:,:]
                    
                    #datcrop = np.flipud(raw).T
                    datcrop = raw
                    
                    mosaic[imap,jmap] = datcrop #datcrop.T #np.flipud(datcrop).T
                    # for 2D map
                    bigimage[imap*sm[2]:(imap+1)*sm[2], jmap*sm[3]:(jmap+1)*sm[3]] = np.flipud(datcrop)
            
            if 0:
                mosaictranspose = mosaic.transpose((0, 3, 1, 2))
                mosaicflat = mosaictranspose.reshape((dimfast * (2 * boxsize_X + 1), dimslow * (2 * boxsize_Y + 1)))

            resdict = {}
            resdict['mosaicdata'] = mosaic
            resdict['singleimage'] = bigimage
            resdict['mosaic_shape'] = mosaic.shape
            return resdict

    @staticmethod
    def _collect_roi_array_single_file(
        index: int,
        roicenter: Tuple[int, int],
        prefix: str,
        folder: str,
        boxsize_X: int,
        boxsize_Y: int,
        ccd_label: str,
    ) -> np.ndarray:
        """Helper function to collect ROI array for a single file
        listindices,
                   itertools.repeat(roicenter),
                   itertools.repeat(prefix),
                   listfolders,
                  itertools.repeat(boxsize_X),
                  itertools.repeat(boxsize_Y),
                   itertools.repeat(CCDLabel)"""
        # Placeholder for actual implementation
        
        return collectroiarray_singlefile(index,roicenter,prefix, folder, boxsize_X, boxsize_Y, ccd_label)




##  Local definition: Pixel monitoring

In [ ]:
class ROICountersWorkflow:
    """Handles ROI counters workflows for DAXM, ascan, etc."""

    def __init__(self, params: Dict[str, Any]):
        self.params = params
        self.boxsize_X = params.get("boxsize_X", 5)
        self.boxsize_Y = params.get("boxsize_Y", 5)
        self.collector = params.get("collector", "roimax")
        self.peaklist = params.get("peaklist")

    def validate_peaklist(self) -> bool:
        """Validate that the peaklist is within detector bounds."""
        if self.peaklist is None:
            logger.error("Peaklist is not set.")
            return False
        framedim = DictLT.dict_CCD[self.params.get("CCDLabel")][0]
        max_X, max_Y = framedim[1], framedim[0]
        for peak in self.peaklist:
            x, y = peak[0], peak[1]
            if (
                x + self.boxsize_X > max_X
                or x - self.boxsize_X < 0
                or y + self.boxsize_Y > max_Y
                or y - self.boxsize_Y < 0
            ):
                logger.error(f"Peak {peak} is too close to the detector border.")
                return False
        return True

    def run_roi_counters_workflow(
        self, nb_cpus: int = 64, list_indices: Optional[List[int]] = None
    ) -> np.ndarray:
        """
        Run the ROI counters workflow to collect intensities.

        Args:
            nb_cpus: Number of CPUs for multiprocessing.
            list_indices: List of image indices to process. If None, uses all indices in params.

        Returns:
            np.ndarray: Array of collected results.
        """
        if not self.validate_peaklist():
            raise ValueError("Invalid peaklist or boxsize.")

        folder = self.params.get("folder")
        prefix = self.params.get("prefix")
        ccd_label = self.params.get("CCDLabel")

        if list_indices is None:
            list_indices = self.params.get("listindices")

        nb_images = len(list_indices)
        max_nb_cpus = cpu_count()
        nb_cpus = min(nb_cpus, max_nb_cpus)
        logger.info(f"Using {nb_cpus} CPUs for {nb_images} images.")

        # Prepare arguments for multiprocessing
        args_roi = zip(
            list_indices,
            itertools.repeat(self.peaklist),
            itertools.repeat(prefix),
            itertools.repeat(folder),
            itertools.repeat(self.boxsize_X),
            itertools.repeat(self.boxsize_Y),
            itertools.repeat(ccd_label),
        )

        # Run multiprocessing
        with Pool(nb_cpus) as pool:
            all_results = list(
                tqdm(
                    pool.starmap(
                        self._collect_roi_single_file, args_roi
                    ),
                    total=nb_images,
                    desc="ROI counters collection progress",
                )
            )

        return np.array(all_results)

    @staticmethod
    def _collect_roi_single_file(
        index: int,
        peaklist: np.ndarray,
        prefix: str,
        folder: str,
        boxsize_X: int,
        boxsize_Y: int,
        ccd_label: str,
    ) -> np.ndarray:
        """TODO !!! Helper function to collect ROI data for a single file."""
        # Placeholder for actual implementation
        # Replace with your logic to load and process a single image
        return np.zeros((len(peaklist), 2 * boxsize_Y + 1, 2 * boxsize_X + 1))

In [ ]:
# ============================================================================
# USE CASE HANDLER
# ============================================================================

class UseCaseHandler:
    """Handles execution of use cases based on experiment parameters."""

    def __init__(self, config: ConfigManager):
        self.config = config
        self.use_cases = {
            "mosaic_2d_map": self._run_mosaic_2d_map,
            "roi_counters_daxm": self._run_roi_counters_daxm,
            "roi_counters_ascan": self._run_roi_counters_ascan,
        }

    def _run_mosaic(self, params: Dict[str, Any]) -> None:
        """Run mosaic workflow for a scans."""
        mosaic_workflow = MosaicWorkflow(params)
        results = mosaic_workflow.run_mosaic_workflow()
        logger.info("Mosaic workflow completed.")
        return results

    def _run_mosaic_2d_map(self, params: Dict[str, Any]) -> None:
        """Run mosaic workflow for a 2D map scans."""
        mosaic_workflow = MosaicWorkflow(params)
        results = mosaic_workflow.run_mosaic_workflow()
        resdict = mosaic_workflow._arrange2D()
        #singleimage_mosaic = resdict.get('singleimage')
        #mosaic_shape = resdict.get('mosaic_shape')
        logger.info("Mosaic workflow completed.")
        return resdict

    def _run_roi_counters_daxm(self, params: Dict[str, Any]) -> None:
        """Run ROI counters workflow for DAXM scans."""
        roi_workflow = ROICountersWorkflow(params)
        results = roi_workflow.run_roi_counters_workflow()
        logger.info("ROI counters workflow for DAXM completed.")
        return results

    def _run_roi_counters_ascan(self, params: Dict[str, Any]) -> None:
        """Run ROI counters workflow for ascan scans."""
        roi_workflow = ROICountersWorkflow(params)
        results = roi_workflow.run_roi_counters_workflow()
        logger.info("ROI counters workflow for ascan completed.")
        return results

    def execute_use_case(self, use_case_name: str, params: Dict[str, Any]) -> Any:
        """
        Execute a use case by name.

        Args:
            use_case_name: Name of the use case to execute.
            params: Dictionary of parameters for the workflow.

        Returns:
            Results of the workflow execution.
        """
        if use_case_name not in self.use_cases:
            raise ValueError(f"Use case '{use_case_name}' not found.")
        return self.use_cases[use_case_name](params)


# visualisation helpers

In [ ]:
# ============================================================================
# VISUALIZATION UTILITIES
# ============================================================================

def plot_mosaic(mosaic_dict_results: np.ndarray, dict_scan_exp= None, vmin=0, vmax=2000) -> None:
    """Plot mosaic data with ROI center highlighted."""

    singleimage_mosaic = mosaic_dict_results.get('singleimage')
    mosaic_shape = mosaic_dict_results.get('mosaic_shape')
    def format_coord(x, y):
        col = int(x)
        row = int(y)
        if col >= 0 and col < singleimage_mosaic.shape[1] and row >= 0 and row < singleimage_mosaic.shape[0]:
            #cnt_idx = fig.gca()
            i, j = col//mosaic_shape.shape[3], row//mosaic_shape.shape[2]
            img_idx = 0+ mosaic_shape.shape[1]*j+i
            return "x=%1.4f, y=%1.4f, imageid=%d" % (x, y,img_idx)
        else:
            return "x=%1.4f, y=%1.4f" % (x, y)


    d = dict_scan_exp
    
    roicenter = d['roicenter']
    print(singleimage_mosaic.shape, )
    figmosaic, axmosaic = plt.subplots(figsize=(8,8))
    axmosaic.format_coord = format_coord
    axmosaic.imshow(singleimage_mosaic, origin='lower', vmax=vmax)
    axmosaic.set_xlabel(f"fastaxis {d['fastaxis']} // pixelX")  # ok for fdscan2d
    axmosaic.set_ylabel(f"slowaxis {d['slowaxis']} // pixelY")
    title = ''
    title += '%s\n'%d['imagefolder']
    title += 'roicenter Det. pixel X,Y = (%d,%d)'%(roicenter[0],roicenter[1])
    axmosaic.set_title(title)
    axmosaic.format_coord = format_coord
    plt.show()


def plot_roi_profile(
    roi_data: np.ndarray, peak_index: int = 0, halfsize: int = 9
) -> None:
    """Plot ROI profile for a given peak."""
    peak_roi_xy = roi_data[peak_index]
    x, y = peak_roi_xy
    xx = np.arange(int(x) - halfsize, int(x) + halfsize + 1)
    icross_x = roi_data[int(y)][xx[0] : xx[-1] + 1]

    fig, ax = plt.subplots()
    ax.plot(xx, icross_x, "-o")
    ax.grid()
    ax.axvline(x, color="k")
    ax.set_title(f"ROI Profile at x,y = {int(x)}, {int(y)}")
    plt.show()


# main script

## with local definitions

In [ ]:
# ============================================================================
# MAIN SCRIPT
# ============================================================================

if __name__ == "__main__":
    # Example usage
    import itertools

    # Initialize configuration
    config = ConfigManager(exp_id="a321217", data_at_esrf=True)

    # this is built from user choice
    # here just few lines of the map
    # number of lines
    nlines = 5
    d = {'scantype': 'map',
         'start_time': pd.Timestamp('2026-07-12 16:01:01'),
         'end_time': pd.Timestamp('2026-07-12 16:30:07'),
         'sample_dataset_scanindex': 'Zr5dimanche_searchgrains_1',
         'fullcommand': 'fscan2d xps -2.06102 0.0005 161 yps -4.43403 0.0005 81 0.1 0.100004',
         'scanindex': '1',
         'motors': 'xps yps',
         'localhdf5file': '/data/visitor/a321217/bm32/20260707/RAW_DATA/Zr5dimanche/Zr5dimanche_searchgrains/Zr5dimanche_searchgrains.h5',
         'imagefolder': '/data/visitor/a321217/bm32/20260707/RAW_DATA/Zr5dimanche/Zr5dimanche_searchgrains/scan0001',
         'endreason': 'SUCCESS',
         'samplename': b'Zr5dimanche',
         'folder': '/data/visitor/a321217/bm32/20260707/RAW_DATA/Zr5dimanche/Zr5dimanche_searchgrains/scan0001',
         'nodeinhdf5file': 'Zr5dimanche/Zr5dimanche_searchgrains/1.1',
         'prefix': 'eiger4m_',
         'suffix': 'h5',
         'listindices': np.arange(81*nlines), #np.arange(13041),
         'nbimagesperline': 81,
         'mapdimensions': (81, nlines), #(81, 161),
         'peaklistfile': None,
         'fastaxis': 'yps',
         'slowaxis': 'xps',
         'collector': 'pixelval',
         'CCDLabel': 'EIGER_4MCdTe'}

    # Example parameters for mosaic workflow: user choice
    mosaic_params = {
        "roicenter": (1334, 1034),
        "boxsize_X": 19,
        "boxsize_Y": 19,
        "collector": "mosaic",
    }

    d.update(mosaic_params)
    # Initialize use case handler
    use_case_handler = UseCaseHandler(config)

    # Execute mosaic workflow
    mosaic_dict_results = use_case_handler.execute_use_case("mosaic_2d_map", d)
    logger.info(f"Mosaic results shape: {mosaic_dict_results['mosaic_shape']}")


In [ ]:
print('mosaic_results', mosaic_dict_results['mosaic_shape'])
plot_mosaic(mosaic_dict_results,d, vmax=250)

## lauetools.workflows version

In [ ]:
# ============================================================================
# MAIN SCRIPT
# ============================================================================

if __name__ == "__main__":
    # Example usage
    import itertools

    # Initialize configuration
    config = wf.ConfigManager(exp_id="a321217", data_at_esrf=True)

    # this is built from user choice
    # here just few lines of the map
    # number of lines
    nlines = 161
    d = {'scantype': 'map',
         'start_time': pd.Timestamp('2026-07-12 16:01:01'),
         'end_time': pd.Timestamp('2026-07-12 16:30:07'),
         'sample_dataset_scanindex': 'Zr5dimanche_searchgrains_1',
         'fullcommand': 'fscan2d xps -2.06102 0.0005 161 yps -4.43403 0.0005 81 0.1 0.100004',
         'scanindex': '1',
         'motors': 'xps yps',
         'localhdf5file': '/data/visitor/a321217/bm32/20260707/RAW_DATA/Zr5dimanche/Zr5dimanche_searchgrains/Zr5dimanche_searchgrains.h5',
         'imagefolder': '/data/visitor/a321217/bm32/20260707/RAW_DATA/Zr5dimanche/Zr5dimanche_searchgrains/scan0001',
         'endreason': 'SUCCESS',
         'samplename': b'Zr5dimanche',
         'folder': '/data/visitor/a321217/bm32/20260707/RAW_DATA/Zr5dimanche/Zr5dimanche_searchgrains/scan0001',
         'nodeinhdf5file': 'Zr5dimanche/Zr5dimanche_searchgrains/1.1',
         'prefix': 'eiger4m_',
         'suffix': 'h5',
         'listindices': np.arange(81*nlines), #np.arange(13041),
         'nbimagesperline': 81,
         'mapdimensions': (81, nlines), #(81, 161),
         'peaklistfile': None,
         'fastaxis': 'yps',
         'slowaxis': 'xps',
         'collector': 'pixelval',
         'CCDLabel': 'EIGER_4MCdTe'}

    # Example parameters for mosaic workflow: user choice
    mosaic_params = {
        "roicenter": (1410, 1595),
        "boxsize_X": 19,
        "boxsize_Y": 19,
        "collector": "mosaic",
    }

    d.update(mosaic_params)
    # Initialize use case handler
    use_case_handler = wf.UseCaseHandler(config)

    # Execute mosaic workflow
    mosaic_dict_results = use_case_handler.execute_use_case("mosaic_2d_map", d)
    wf.logger.info(f"Mosaic results shape: {mosaic_dict_results['mosaic_shape']}")


In [ ]:
print('mosaic_results', mosaic_dict_results['mosaic_shape'])
plot_mosaic(mosaic_dict_results,d, vmax=250)

## roi counters

In [ ]:
if 0: # TODO !!!# Example parameters for ROI counters workflow
    if __name__ == "__main__":
        # Example usage
        import itertools
    
        # Initialize configuration
        config = ConfigManager(exp_id="a321217", data_at_esrf=True)
    
        # this is built from user choice
        # here just few lines of the map
        # number of lines
        nlines = 13
        d = {'scantype': 'map',
             'start_time': pd.Timestamp('2026-07-12 16:01:01'),
             'end_time': pd.Timestamp('2026-07-12 16:30:07'),
             'sample_dataset_scanindex': 'Zr5dimanche_searchgrains_1',
             'fullcommand': 'fscan2d xps -2.06102 0.0005 161 yps -4.43403 0.0005 81 0.1 0.100004',
             'scanindex': '1',
             'motors': 'xps yps',
             'localhdf5file': '/data/visitor/a321217/bm32/20260707/RAW_DATA/Zr5dimanche/Zr5dimanche_searchgrains/Zr5dimanche_searchgrains.h5',
             'imagefolder': '/data/visitor/a321217/bm32/20260707/RAW_DATA/Zr5dimanche/Zr5dimanche_searchgrains/scan0001',
             'endreason': 'SUCCESS',
             'samplename': b'Zr5dimanche',
             'folder': '/data/visitor/a321217/bm32/20260707/RAW_DATA/Zr5dimanche/Zr5dimanche_searchgrains/scan0001',
             'nodeinhdf5file': 'Zr5dimanche/Zr5dimanche_searchgrains/1.1',
             'prefix': 'eiger4m_',
             'suffix': 'h5',
             'listindices': np.arange(81*nlines), #np.arange(13041),
             'nbimagesperline': 81,
             'mapdimensions': (81, nlines), #(81, 161),
             'peaklistfile': None,
             'fastaxis': 'yps',
             'slowaxis': 'xps',
             'collector': 'pixelval',
             'CCDLabel': 'EIGER_4MCdTe'}
    
        
    
        roi_params = {
            "scantype": "daxm",
            "folder": os.path.join(config.experiment_folder, "A45/A45_line1daxms/scan0005"),
            "prefix": "img_",
            "listindices": np.arange(0, 420),
            "mapdimensions": (420, 1),
            "CCDLabel": "sCMOS",
            "peaklist": np.random.randint(0, 2000, size=(10, 2)),  # Example peaklist
            "boxsize_X": 3,
            "boxsize_Y": 1,
            "collector": "roimax",
        }
    
        # Execute ROI counters workflow
        roi_results = use_case_handler.execute_use_case("roi_counters_daxm", roi_params)
        logger.info(f"ROI counters results shape: {roi_results.shape}")